In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
from scipy.stats import pearsonr
import sys
sys.path.insert(0,'..')
from scripts.models.pod.model import RampPOD
warnings.filterwarnings('ignore')
pplt.rc.update({
    'tick.minor':False,'savefig.dpi':300,
    'font.size':9,'label.size':9,'tick.labelsize':9,
    'legend.fontsize':9,'leftlabelsize':9,'toplabelsize':9,
    'leftlabel.weight':'normal','toplabel.weight':'normal'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELSDIR  = CONFIGS['filepaths']['models']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELS     = CONFIGS['experiments']
NNCONFIG   = MODELS['nn']
SRCONFIG   = MODELS['sr']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']
YEARS      = CONFIGS['domain']['years']  # 2000-2020, all 21 JJA seasons
MONTHS     = CONFIGS['domain']['months']

url = 'https://psl.noaa.gov/data/correlation/nina34.anom.data'
raw = pd.read_csv(url,sep=r'\s+',header=None,skiprows=1,engine='python')
raw = raw[pd.to_numeric(raw[0],errors='coerce').notna()]
raw = raw.iloc[:,:13].apply(pd.to_numeric,errors='coerce')
nino = {}
for _,row in raw.iterrows():
    year = int(row[0])
    if YEARS[0]<=year<=YEARS[-1]:
        jja = np.nanmean(row[MONTHS].to_numpy())
        if np.isfinite(jja):
            nino[year] = float(jja)
print(f'NINO3.4 loaded for {len(nino)} years: {sorted(nino)[:3]}...{sorted(nino)[-3:]}')

In [ ]:
def load_all_splits(var):
    pieces = []
    for split in ['train','valid','test']:
        path = os.path.join(SPLITSDIR,f'{split}.h5')
        with xr.open_dataset(path,engine='h5netcdf') as ds:
            if var in ds:
                pieces.append(ds[var].load())
    return xr.concat(pieces,dim='time')

def annual_q95(da,years):
    slices = []
    for year in years:
        mask = da.time.dt.year==year
        if not mask.any():
            continue
        q = da.sel(time=mask).quantile(0.95,dim='time',skipna=True)
        if 'quantile' in q.coords:
            q = q.drop_vars('quantile')
        slices.append(q.expand_dims(year=[year]))
    return xr.concat(slices,dim='year')

def pearson_map(q95da,ninodict):
    years     = q95da.year.values
    nino_arr  = np.array([ninodict.get(int(y),np.nan) for y in years],dtype=float)
    q95values = q95da.values.astype(float)
    nlat,nlon = q95values.shape[1],q95values.shape[2]
    rmap = np.full((nlat,nlon),np.nan)
    pmap = np.full((nlat,nlon),np.nan)
    for i in range(nlat):
        for j in range(nlon):
            v  = q95values[:,i,j]
            ok = np.isfinite(v)&np.isfinite(nino_arr)
            if ok.sum()>=5:
                rmap[i,j],pmap[i,j] = pearsonr(nino_arr[ok],v[ok])
    coords = {'lat':q95da.lat,'lon':q95da.lon}
    return (xr.DataArray(rmap,dims=['lat','lon'],coords=coords),
            xr.DataArray(pmap,dims=['lat','lon'],coords=coords))

def fit_empirical_qm(a,b,nquantiles=2000):
    apos,bpos = a[a>0],b[b>0]
    quantiles = np.linspace(0,1,nquantiles)
    aq = np.maximum.accumulate(np.quantile(apos,quantiles) if len(apos)>0 else np.array([0.]))
    bq = np.maximum.accumulate(np.quantile(bpos,quantiles) if len(bpos)>0 else np.array([0.]))
    def qm(x):
        x   = np.maximum(np.asarray(x,float),0)
        out = np.zeros_like(x)
        out[x>0] = np.interp(x[x>0],aq,bq,left=bq[0],right=bq[-1])
        return out
    return qm

def load_kernel_weights(weightsfrom):
    wlist = []
    for seed in NNCONFIG['seeds']:
        wpath = os.path.join(WEIGHTSDIR,f'{weightsfrom}_{seed}_weights.nc')
        with xr.open_dataset(wpath,engine='h5netcdf') as wds:
            wlist.append(wds['k'].load())
    return xr.concat(wlist,dim='seed').mean('seed')

def kernel_integrate_fields(fieldvars,splitvars,weightsmean):
    dsig_da = splitvars['dsig']
    if 'time' in dsig_da.dims:
        dsig_da = dsig_da.mean(dim='time')
    dsig   = np.asarray(dsig_da.transpose('sig').values).reshape(-1)
    result = {}
    for var in fieldvars:
        da        = splitvars[var]
        spacedims = [d for d in da.dims if d != 'sig']
        arr       = da.transpose(*spacedims,'sig').values
        w         = np.asarray(weightsmean.sel(field=var).values).reshape(-1)
        integ     = (arr.reshape(-1,arr.shape[-1])*w[None,:]*dsig[None,:]).sum(axis=1)
        coords    = {d:da.coords[d] for d in spacedims if d in da.coords}
        result[var] = xr.DataArray(integ.reshape(arr.shape[:-1]),dims=spacedims,coords=coords)
    return result

def load_nn_allyears(name):
    pieces = []
    for split in ['train','valid','test']:
        path = os.path.join(PREDSDIR,f'{name}_{split}_predictions.nc')
        if not os.path.exists(path):
            continue
        with xr.open_dataset(path) as ds:
            pred = ds['tp'].load()
            if 'seed' in pred.dims:
                pred = pred.mean('seed')
            pieces.append(pred)
    return xr.concat(pieces,dim='time') if pieces else None

In [ ]:
trainds = xr.open_dataset(os.path.join(SPLITSDIR,'train.h5'),engine='h5netcdf')[['tp','pr']].load()
validds = xr.open_dataset(os.path.join(SPLITSDIR,'valid.h5'),engine='h5netcdf')[['tp','pr']].load()
era5  = xr.concat([trainds.tp,validds.tp],dim='time').values.ravel()
imerg = xr.concat([trainds.pr,validds.pr],dim='time').values.ravel()
mask  = np.isfinite(era5)&np.isfinite(imerg)
qmfunction = fit_empirical_qm(era5[mask],imerg[mask])
del trainds,validds,era5,imerg,mask
print('QM function fitted on train+valid (2000-2017)')

In [ ]:
# IMERG (ground truth) — all years
imergall  = load_all_splits('pr')
q95imerg  = annual_q95(imergall,YEARS)
rimerg,pimerg = pearson_map(q95imerg,nino)
del imergall,q95imerg
print('IMERG done')

# POD — all years
bl = load_all_splits('bl')
with np.load(os.path.join(MODELSDIR,'pod','pod_bl.npz')) as d:
    pod = RampPOD(float(d['alpha']),float(d['xcrit']))
predpod   = xr.DataArray(pod.forward(bl).reshape(bl.shape),dims=bl.dims,coords=bl.coords)
qmpod     = xr.apply_ufunc(qmfunction,predpod,vectorize=False)
q95pod    = annual_q95(qmpod,YEARS)
rpod,ppod = pearson_map(q95pod,nino)
del bl,predpod,qmpod,q95pod
print('POD done')

# Baseline NN and Parametric Kernel NN — load saved predictions for all splits
nn_refs = {}
for nn_name in ['base_all','param_gauss_all']:
    pred = load_nn_allyears(nn_name)
    if pred is None:
        print(f'{nn_name}: no saved predictions found, skipping')
        continue
    qmpred   = xr.apply_ufunc(qmfunction,pred,vectorize=False)
    q95pred  = annual_q95(qmpred,YEARS)
    r,p      = pearson_map(q95pred,nino)
    nn_refs[nn_name] = (r,p)
    del pred,qmpred,q95pred
    print(f'{nn_name} done')

In [ ]:
piecesnorm = []
for split in ['train','valid','test']:
    normpath = os.path.join(SPLITSDIR,f'norm_{split}.h5')
    with xr.open_dataset(normpath,engine='h5netcdf') as ds:
        piecesnorm.append({v:ds[v].load() for v in ds.data_vars})

allvars = sorted(set().union(*[set(p.keys()) for p in piecesnorm]))
svars   = {v:xr.concat([p[v] for p in piecesnorm if v in p],dim='time') for v in allvars}
if 'dsig' not in svars:
    with xr.open_dataset(os.path.join(SPLITSDIR,'train.h5'),engine='h5netcdf') as ds:
        svars['dsig'] = ds['dsig'].load()

with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    srstats = json.load(f)
srrefda  = svars[CONFIGS['variables']['target']]

# Pre-compute kernel-integrated field variables (shared across all SR runs using same weightsfrom)
cached_integ = {}
for runname,runconfig in SRCONFIG['runs'].items():
    weightsfrom = runconfig.get('weightsfrom')
    fieldvars   = runconfig['fieldvars']
    key         = (weightsfrom,tuple(fieldvars))
    if key not in cached_integ:
        print(f'Kernel-integrating {fieldvars} using weights from {weightsfrom}...')
        cached_integ[key] = kernel_integrate_fields(fieldvars,svars,load_kernel_weights(weightsfrom))
print('SR features ready')

In [ ]:
def eval_sr_complexity_sweep(runname,runconfig):
    csvpath = os.path.join(MODELSDIR,'sr',f'{runname}_equations.csv')
    if not os.path.exists(csvpath):
        print(f'{runname}: no equations CSV, skipping')
        return {}
    eqdf        = pd.read_csv(csvpath)
    fieldvars   = runconfig['fieldvars']
    localvars   = runconfig.get('localvars',[])
    weightsfrom = runconfig.get('weightsfrom')
    srinteg     = cached_integ[(weightsfrom,tuple(fieldvars))]
    ns          = {'Abs':np.abs,'abs':np.abs,'sqrt':np.sqrt,'exp':np.exp,
                   'log':np.log,'sin':np.sin,'cos':np.cos,'__builtins__':{}}
    for v,da in srinteg.items():
        ns[v] = da.transpose('time','lat','lon').values.ravel()
    for v in localvars:
        if v not in svars:
            continue
        da    = svars[v]
        extra = [d for d in da.dims if d not in ('time','lat','lon')]
        if extra:
            da = da.mean(dim=extra)
        if 'time' in da.dims and da.sizes['time'] != srrefda.sizes['time']:
            da = da.mean(dim='time')
        ns[v] = da.broadcast_like(srrefda).transpose('time','lat','lon').values.ravel()
    tshape  = srrefda.values.ravel().shape
    results = {}
    for _,row in eqdf.iterrows():
        eqstr = str(row['sympy_format'])
        compl = int(row['complexity'])
        loss  = float(row['loss'])
        try:
            ypredn  = np.asarray(eval(eqstr,ns),dtype=float)
            if ypredn.shape == ():
                ypredn = np.full(tshape,float(ypredn))
            ypredmm = np.maximum(np.expm1(np.clip(ypredn,None,20)*srstats['tp_std']+srstats['tp_mean']),0.0)
            srda    = xr.DataArray(ypredmm.reshape(srrefda.shape),dims=srrefda.dims,coords=srrefda.coords)
            qmsr    = xr.apply_ufunc(qmfunction,srda,vectorize=False)
            r,p     = pearson_map(annual_q95(qmsr,YEARS),nino)
            results[compl] = dict(r=r,p=p,loss=loss,eq=eqstr)
            print(f'  {runname} complexity={compl}: done')
        except Exception as e:
            print(f'  {runname} complexity={compl}: failed ({e})')
    return results

sr_sweep = {}
for runname,runconfig in SRCONFIG['runs'].items():
    print(f'Sweeping {runname}...')
    sr_sweep[runname] = eval_sr_complexity_sweep(runname,runconfig)
del svars

In [ ]:
NNCAPTION = {'base_all':'Baseline NN','param_gauss_all':'Gaussian Kernel NN'}

for runname,compl_dict in sr_sweep.items():
    if not compl_dict:
        continue
    complexities = sorted(compl_dict)
    best_c = min(complexities,key=lambda c: compl_dict[c]['loss'])

    ref_panels = []
    ref_panels.append((rimerg,pimerg,'IMERG V06'))
    ref_panels.append((rpod,ppod,'POD'))
    for nn_name,caption in NNCAPTION.items():
        if nn_name in nn_refs:
            ref_panels.append((*nn_refs[nn_name],caption))

    sr_panels = [(compl_dict[c]['r'],compl_dict[c]['p'],
                  f'SR c={c}' + (' *' if c==best_c else ''))
                 for c in complexities]

    all_panels = ref_panels + sr_panels
    ncols      = min(len(all_panels),6)
    nrows      = int(np.ceil(len(all_panels)/ncols))
    fig,axs    = pplt.subplots(nrows=nrows,ncols=ncols,proj='cyl',refwidth=1.8)
    axs.format(coast=True,borders=True,
               latlim=(LATRANGE[0],LATRANGE[1]),lonlim=(LONRANGE[0],LONRANGE[1]),
               latlines=5,lonlines=5)
    mappable = None
    for ax,(r,p,title) in zip(axs,all_panels):
        ax.format(title=title)
        if r is None:
            ax.format(visible=False)
            continue
        m = ax.pcolormesh(r.lon.values,r.lat.values,r.values,
                          cmap='ColdHot',vmin=-0.7,vmax=0.7,N=15)
        if mappable is None:
            mappable = m
        if p is not None:
            pmask = p.values<0.05
            if pmask.any():
                lons2d,lats2d = np.meshgrid(r.lon.values,r.lat.values)
                ax.scatter(lons2d[pmask],lats2d[pmask],c='k',s=1,zorder=5,linewidths=0)
    # hide any unused axes
    for ax in axs[len(all_panels):]:
        ax.format(visible=False)
    fig.colorbar(mappable,loc='b',
                 label='Pearson $r$  (JJA NINO3.4 Anomalies vs. JJA 95th-percentile Precipitation, 2000-2020)')
    fig.format(suptitle=f'{runname} — all Pareto complexity levels  (* = min-loss equation)')
    pplt.show()

In [ ]:
imerg_sig = pimerg.values < 0.05
print(f'IMERG significant pixels (p<0.05): {imerg_sig.sum()} / {imerg_sig.size}')

# Reference scalars
def mean_r_sig(r):
    return float(np.nanmean(r.values[imerg_sig]))

ref_scalars = {'IMERG':mean_r_sig(rimerg),'POD':mean_r_sig(rpod)}
for nn_name,caption in NNCAPTION.items():
    if nn_name in nn_refs:
        ref_scalars[caption] = mean_r_sig(nn_refs[nn_name][0])

fig,ax = pplt.subplots(refwidth=5,refheight=2.5)
colors = {'IMERG':'gray6','POD':'gray6','Baseline NN':'blue6','Gaussian Kernel NN':'red6'}
for label,val in ref_scalars.items():
    ax.axhline(val,color=colors.get(label,'gray6'),
               linestyle='--' if label=='IMERG' else ':',lw=1.2,label=label)

for runname,compl_dict in sr_sweep.items():
    if not compl_dict:
        continue
    complexities = sorted(compl_dict)
    best_c       = min(complexities,key=lambda c: compl_dict[c]['loss'])
    mean_rs      = [mean_r_sig(compl_dict[c]['r']) for c in complexities]
    ax.plot(complexities,mean_rs,color='green7',marker='o',ms=5,lw=1.5,label=runname)
    ax.scatter([best_c],[mean_r_sig(compl_dict[best_c]['r'])],
               color='green7',marker='*',s=100,zorder=5,label='min-loss equation')

ax.format(xlabel='SR Equation Complexity',
          ylabel='Mean Pearson $r$ over IMERG-significant pixels',
          title='ENSO Coherence vs. SR Complexity  (all 21 JJA seasons, 2000-2020)',
          grid=True)
ax.legend(loc='right',ncols=1)
pplt.show()